# QC: Cellpose segmentation mask vs DAPI

Two ways to validate that **Cellpose** regions align with **nuclear (DAPI) signal**:

1. **Visual (Napari)** — overlay the DAPI image channel and the label mask; pan/zoom to look for mis-segmentation, boundary drift, or dim nuclei outside mask.
2. **Quantitative (this notebook or `scripts/qc_mask_vs_dapi.py`)** — at a **downsampled pyramid level** (same for image + mask), compare DAPI intensity **inside** label `>0` vs **background** and plot histograms.

**Default paths** (edit below if your pipeline uses different files):
- Image: `data/CellDIVE_SLIDE-045.zarr` — DAPI is resolved from `.zattrs` (primary DAPI, not DAPI2/R06 when possible).
- Mask: `output/cellpose_output/cellpose_masks_dapi_only_9tiles.zarr` — must match the mask used to build the cell matrix.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.qc_mask_vs_dapi import dapi_channel_index, run_qc

CELLDIVE_ZARR = PROJECT_ROOT / "data" / "CellDIVE_SLIDE-045.zarr"
MASK_ZARR     = PROJECT_ROOT / "output" / "cellpose_output" / "cellpose_masks_dapi_only_9tiles.zarr"
OUT_FIG       = PROJECT_ROOT / "output" / "figures" / "mask_vs_dapi_qc.png"

# Pyramid level: 0 = full res (slow, huge RAM). 3–4 is usually enough for global QC.
PYRAMID_LEVEL = 4
# 0 = use every pixel at that level; set e.g. 5_000_000 to subsample for speed on huge levels
MAX_PIXELS   = 0

print("DAPI channel index (from .zattrs):", dapi_channel_index(CELLDIVE_ZARR))

DAPI channel index (from .zattrs): 0


In [2]:
assert CELLDIVE_ZARR.is_dir(), f"Missing: {CELLDIVE_ZARR}"
assert MASK_ZARR.is_dir(),     f"Missing: {MASK_ZARR}"

stats = run_qc(
    CELLDIVE_ZARR,
    MASK_ZARR,
    level=PYRAMID_LEVEL,
    max_pixels=MAX_PIXELS,
    seed=0,
    out_figure=OUT_FIG,
)

for k, v in stats.items():
    if k in ("percentile_inside", "percentile_outside"):
        print(f"{k}:", v)
    else:
        print(f"{k}: {v}")

level: 4
dapi_channel_index: 0
shape_2d: (4039, 4622)
n_pixels_total: 18668258
subsampled: 0
n_sample_used: 18668258
frac_mask_fg: 0.1880819838680181
mean_dapi_inside: 3056.0107421875
mean_dapi_outside: 769.0592651367188
median_dapi_inside: 2511.0
median_dapi_outside: 198.0
ratio_fg_over_bg: 3.9736999223905336
percentile_inside: {'p1': 299.0, 'p5': 708.0, 'p10': 962.0, 'p25': 1539.0, 'p50': 2511.0, 'p75': 3963.0, 'p90': 5848.0, 'p99': 10693.0}
percentile_outside: {'p1': 98.0, 'p5': 98.0, 'p10': 98.0, 'p25': 129.0, 'p50': 198.0, 'p75': 1127.0, 'p90': 2016.0, 'p99': 4889.0}
frac_fg_below_global_p5: 0.0
figure: /home/steve/Projects/HeLab/BladderDIVE/output/figures/mask_vs_dapi_qc.png


### How to read the quick stats

- **mean_dapi_inside** should be **higher** than **mean_dapi_outside** if the mask carves out cell bodies/nuclei on DAPI-driven segmentation. A **ratio** $\gg 1$ is expected for DAPI-nuclear tissue.
- **frac_fg_below_global_p5** — fraction of mask **pixels** (not cells) whose DAPI is below the **global** 5th percentile. This flags mass of segmentation sitting on very dim DAPI. Per-cell gating in `analyze_cell_types` uses a different (cell-aggregated) statistic.
- This does **not** replace biology review: if something fails, check **Napari** and/or re-run **Cellpose** on a region.

Figure: `output/figures/mask_vs_dapi_qc.png` (histogram of DAPI inside mask vs background at chosen pyramid level).

---
## Visual check (Napari, optional)

Requires: `napari` + `dask` in the environment (e.g. `napari-env`).

From the repo root:

**Use the annotated H5AD** — `output/celldive_protein_matrix.h5ad` (protein matrix only) has **no** `cell_type`
column. Load **`celldive_protein_matrix_celltypes.h5ad`** after running the cell-typing notebook.

```bash
python scripts/napari_load_image_adata.py \
  output/celldive_protein_matrix_celltypes.h5ad cell_type --mode both
# Loads the CellDIVE pyramid + mask outline. Turn off all image channels except DAPI to check alignment.
```

Or in a notebook (GUI session):

```python
from pathlib import Path
import anndata as ad
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from scripts.napari_load_image_adata import open_napari_with_adata

adata = ad.read_h5ad(PROJECT_ROOT / "output" / "celldive_protein_matrix_celltypes.h5ad")
open_napari_with_adata(adata, "cell_type", project_root=PROJECT_ROOT, display_mode="both", add_mask_layer=True)
import napari; napari.run()
```

Turn other channels off and leave **DAPI** on to confirm boundaries follow nuclear signal.

**Read intensity at a pixel in Napari:** select the **Image** layer (e.g. the DAPI channel) in the layer
list, then move the **cursor** over the canvas. The **status bar** (bottom of the main window) usually
shows coordinates and the **intensity** at the cursor for the **active** layer. For **multiscale** images,
the value is from the **resolution level** you are currently viewing. If nothing appears, single-click
the image layer to focus it, and ensure the status bar is visible (*View* menu may have *Toggle status
bar* / *status bar* depending on version). Some installations also ship a *Console* (*Window → Console*)
where you can query layers programmatically.

---
**CLI (no notebook):**  
`python scripts/qc_mask_vs_dapi.py --level 4`

In [3]:
from pathlib import Path
import anndata as ad
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.napari_load_image_adata import open_napari_with_adata

# Must be the annotated H5AD (has obs['cell_type']). Plain celldive_protein_matrix.h5ad has no cell_type.
H5AD = PROJECT_ROOT / "output" / "celldive_protein_matrix_celltypes.h5ad"
if not H5AD.exists():
    raise FileNotFoundError(f"Not found: {H5AD}\n  Run analyze_cell_types_from_markers.ipynb or use a column that exists in your h5ad.")

adata = ad.read_h5ad(H5AD)
open_napari_with_adata(adata, "cell_type", project_root=PROJECT_ROOT, display_mode="both", add_mask_layer=True)
import napari; napari.run()

Loaded Cellpose segmentation as Labels layer (multiscale pyramid, contour=1)
Loaded cells as filled Labels (scale level 2)
Loaded 100000 cells as 20 point layers
